# Rules

- Keep English Terms in English
    > for example: " ببساطة Backpropagation هلق لح نشرح شو هوي ال "

- Write as you hear
    > ❌ ليك ما تروحي لعنده لهذا'
    
    > ✅ ليكي ما تروحي لعندو لهاد'
    
- No Diacritics (Tashkeel)

- Write numbers the same way they are being said : "خمسة وعشرين" instead of "25"

- keep filler words like "اممم"

In [1]:
 pip install --upgrade transformers accelerator librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 100.5 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.

In [2]:
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from pathlib import Path
import csv
import torch
from transformers import pipeline
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [3]:
# -------------------------------------------------
# Initialize Whisper Large Turbo Pipeline
# -------------------------------------------------
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)


print(f"✅ Model loaded successfully on {device.upper()}.\n")

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

... (4 outputs left out)

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

✅ Model loaded successfully on CUDA:0.



In [4]:
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


In [5]:
import csv
from pathlib import Path
from pydub import AudioSegment
from pydub.silence import detect_nonsilent

def smart_split_dataset(
    input_folder,
    asr_pipeline,
    speaker_id,
    annotator_id,  
    speaker_metadata,  # Expects dict: {"sex": "M", "dialect": "Shami"}
    output_folder="dataset",
    target_length_sec=20,
    min_silence_len=300,
    silence_offset_db=16,
    keep_silence=250
):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(exist_ok=True)

    speaker_metadata = speaker_metadata or {}

    # =====================================================
    # 1. Load the Lecture Manifest File
    # =====================================================
    manifest_path = input_folder / "manifest.csv"
    lecture_manifest = {}
    
    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if "file_name" in row:
                    fname = row["file_name"]
                    # Store everything except the filename key as metadata
                    lecture_manifest[fname] = {k: v for k, v in row.items() if k != "file_name"}
        print(f"📖 Loaded lecture manifest with {len(lecture_manifest)} file entries.")
    else:
        print("⚠ No manifest.csv found in input folder. Lectures will not have unique metadata.")
        return

    # =====================================================
    # 2. Save/Update speaker metadata (unchanged)
    # =====================================================
    speakers_csv = output_folder / "speakers.csv"
    existing_speaker_ids = set()
    speaker_fields = ["speaker_id"] + list(speaker_metadata.keys())

    if speakers_csv.exists():
        with open(speakers_csv, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            existing_speaker_ids = {row["speaker_id"] for row in reader if "speaker_id" in row}

    if speaker_id not in existing_speaker_ids:
        write_header = not speakers_csv.exists()
        with open(speakers_csv, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=speaker_fields)
            if write_header:
                writer.writeheader()
            writer.writerow({"speaker_id": speaker_id, **speaker_metadata})

    # Supported formats
    supported_formats = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac"}
    audio_files = sorted([f for f in input_folder.iterdir() if f.suffix.lower() in supported_formats])

    if not audio_files:
        print("❌ No audio files found.")
        return

    # =====================================================
    # 3. Process each lecture
    # =====================================================
    for lecture_idx, audio_path in enumerate(audio_files):
        lecture_id = f"L{lecture_idx:03d}"
        print("=" * 60)
        print(f"Processing {audio_path.name}")

        # Extract specific metadata for this exact audio file
        current_lecture_metadata = lecture_manifest.get(audio_path.name, {"topic": "Unknown", "audio_quality": "Unknown"})

        speaker_folder = output_folder / speaker_id
        lecture_folder = speaker_folder / f"lecture_{lecture_idx:03d}"
        chunks_folder = lecture_folder / "chunks"
        chunks_folder.mkdir(parents=True, exist_ok=True)

        # Load + standardize audio
        audio = AudioSegment.from_file(audio_path, parameters=["-ar", "16000", "-ac", "1"])
        original_audio_path = lecture_folder / "original.wav"
        audio.export(original_audio_path, format="wav")

        print("Analyzing speech regions...")
        speech_ranges = detect_nonsilent(audio, min_silence_len=min_silence_len, silence_thresh=audio.dBFS - silence_offset_db)

        if not speech_ranges:
            print("⚠ No speech detected.")
            continue

        merged_ranges = []
        current_start, current_end = speech_ranges[0][0], speech_ranges[0][1]
        target_ms = target_length_sec * 1000

        for start, end in speech_ranges[1:]:
            if (end - current_start) <= target_ms:
                current_end = end
            else:
                merged_ranges.append((current_start, current_end))
                current_start, current_end = start, end
        merged_ranges.append((current_start, current_end))

        metadata_rows = []

        for chunk_idx, (start_ms, end_ms) in enumerate(merged_ranges):
            padded_start = max(0, start_ms - keep_silence)
            padded_end = min(len(audio), end_ms + keep_silence)
            chunk_audio = audio[padded_start:padded_end]
            
            filename_base = f"{speaker_id}_{lecture_id}_C{chunk_idx:03d}"
            audio_filename = f"{filename_base}.wav"

            chunk_path = chunks_folder / audio_filename


            chunk_audio.export(chunk_path, format="wav")

            try:
                result = asr_pipeline(str(chunk_path))
                transcription = result.get("text", "").strip()
            except Exception as e:
                print(f"⚠ Transcription failed for {audio_filename}: {e}")
                transcription = ""


            duration_sec = (padded_end - padded_start) / 1000

            # Merge everything smoothly into the row entry
            row_entry = {
                "chunk_file": audio_filename,
                "start_sec": round(padded_start / 1000, 2),
                "end_sec": round(padded_end / 1000, 2),
                "duration_sec": round(duration_sec, 2),
                "text": transcription,
                "speaker_id": speaker_id,
                "annotator_id": annotator_id,  # <-- Added to row data
                **speaker_metadata,
                **current_lecture_metadata  # <-- Dynamic per lecture file!
            }
            metadata_rows.append(row_entry)

        # Save metadata out to chunk_metadata.csv
        metadata_csv = lecture_folder / "chunk_metadata.csv"
        # Included "annotator_id" inside base_fields to track data ownership per chunk
        base_fields = ["chunk_file", "start_sec", "end_sec", "duration_sec", "text", "speaker_id", "annotator_id"]
        chunk_csv_fields = base_fields + list(speaker_metadata.keys()) + list(current_lecture_metadata.keys())

        with open(metadata_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=chunk_csv_fields)
            writer.writeheader()
            writer.writerows(metadata_rows)

        print(f"✅ Finished lecture {lecture_id}\n")

In [6]:
# smart_split_dataset(
#     input_folder="/kaggle/input/datasets/mahdiat11/wajeez-test-sample/Test_Samples/sample_1",
#     asr_pipeline= "asr_pipeline",
#     speaker_id="SPK_001",
#     speaker_metadata = {"sex": "M", "dialect": "Shami"}
# )

## د. ابي صندوق

In [7]:
smart_split_dataset(
    input_folder="/kaggle/input/datasets/mahdiat11/damascus-university-audio-mahdi/Audio_Data/ابي_صندوق",
    asr_pipeline= asr_pipeline,
    speaker_id="SPK-0002",
    annotator_id=2,
    speaker_metadata = {"sex": "M", "dialect": "Shami"}
)

📖 Loaded lecture manifest with 1 file entries.
Processing الخوارزميات 1 رقم 3.m4a
Analyzing speech regions...


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

✅ Finished lecture L000



## د. عمار النحاس

In [8]:
smart_split_dataset(
    input_folder="/kaggle/input/datasets/mahdiat11/damascus-university-audio-mahdi/Audio_Data/عمار_النحاس",
    asr_pipeline= asr_pipeline,
    speaker_id="SPK-0001",
    annotator_id=2,
    speaker_metadata = {"sex": "M", "dialect": "Shami"}
)

📖 Loaded lecture manifest with 2 file entries.
Processing 30بيانيات.mp3
Analyzing speech regions...
✅ Finished lecture L000

Processing fuzzy.aac
Analyzing speech regions...
✅ Finished lecture L001



## د. مثنى خضر

In [9]:
smart_split_dataset(
    input_folder="/kaggle/input/datasets/mahdiat11/damascus-university-audio-mahdi/Audio_Data/مثنى_خضر",
    asr_pipeline= asr_pipeline,
    speaker_id="SPK-0003",
    annotator_id=2,
    speaker_metadata = {"sex": "M", "dialect": "Sahli"}
)

📖 Loaded lecture manifest with 1 file entries.
Processing OS 2.m4a
Analyzing speech regions...
⚠ Transcription failed for SPK-0003_L000_C075.wav: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.
⚠ Transcription failed for SPK-0003_L000_C089.wav: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.
⚠ Transcription failed for SPK-0003_L000_C125.wav: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True

## د. هدى حبش

In [10]:
smart_split_dataset(
    input_folder="/kaggle/input/datasets/mahdiat11/damascus-university-audio-mahdi/Audio_Data/هدى_حبش",
    asr_pipeline= asr_pipeline,
    speaker_id="SPK-0004",
    annotator_id=2,
    speaker_metadata = {"sex": "F", "dialect": "Shami"}
)

📖 Loaded lecture manifest with 1 file entries.
Processing 2 احتمالات واحصاء قسم اول.m4a
Analyzing speech regions...
⚠ Transcription failed for SPK-0004_L000_C000.wav: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.
⚠ Transcription failed for SPK-0004_L000_C099.wav: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.
⚠ Transcription failed for SPK-0004_L000_C135.wav: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `r

Zip to download

In [11]:
!zip -r dataset.zip /kaggle/working/dataset

  adding: kaggle/working/dataset/ (stored 0%)
  adding: kaggle/working/dataset/SPK-0001/ (stored 0%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/ (stored 0%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/ (stored 0%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/SPK-0001_L001_C062.wav (deflated 52%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/SPK-0001_L001_C001.wav (deflated 59%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/SPK-0001_L001_C086.wav (deflated 58%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/SPK-0001_L001_C087.txt (deflated 58%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/SPK-0001_L001_C028.wav (deflated 47%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/SPK-0001_L001_C079.txt (deflated 57%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/SPK-0001_L001_C066.txt (deflated 40%)
  adding: kaggle/working/dataset/SPK-0001/lecture_001/chunks/SPK-

In [12]:
# from IPython.display import FileLink

# FileLink("./dataset.zip")

In [14]:
from huggingface_hub import login

login()

In [15]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id="noone-m/Damascus_University_Audio_Mahdi",
    repo_type="dataset",
    exist_ok=True
)

RepoUrl('https://huggingface.co/datasets/noone-m/Damascus_University_Audio_Mahdi', endpoint='https://huggingface.co', repo_type='dataset', repo_id='noone-m/Damascus_University_Audio_Mahdi')

In [16]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="dataset.zip",
    path_in_repo="dataset.zip",
    repo_id="noone-m/Damascus_University_Audio_Mahdi",
    repo_type="dataset"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/noone-m/Damascus_University_Audio_Mahdi/commit/131f80729c9e1c9eafb7a396b65f58cd214f1ab4', commit_message='Upload dataset.zip with huggingface_hub', commit_description='', oid='131f80729c9e1c9eafb7a396b65f58cd214f1ab4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/noone-m/Damascus_University_Audio_Mahdi', endpoint='https://huggingface.co', repo_type='dataset', repo_id='noone-m/Damascus_University_Audio_Mahdi'), pr_revision=None, pr_num=None)